# EDA и бинарная классификация на датасете «Титаник»

**Дисциплина:** Машинное обучение и ИИ  
**Модуль 1:** Введение в Data Science и EDA  
**Датасет:** Titanic — Machine Learning from Disaster  
**Выполнил:** Дремин Иларион Алексеевич  
**Группа:** АСОиУб-23-2  
**Преподаватель:** Игорь Спирин  
**Ссылка на репозиторий:** `будет добавлена после создания GitHub`  
**Путь к модулю:** `module-1-titanic/notebook.ipynb`  
**Дата сдачи:** 2026-09-29


## Введение

**Задача:** решить задачу бинарной классификации — предсказать, выжил ли пассажир Titanic (`0` — не выжил, `1` — выжил).

**ML-проект** — это последовательность этапов: получение и понимание данных → разведочный анализ (EDA) → подготовка и создание признаков → обучение моделей → оценка качества → интерпретация и сохранение результата.

**Почему это классификация, а не регрессия:** целевая переменная `Survived` принимает два дискретных значения, поэтому модель должна выбрать один из двух классов. При регрессии предсказывается непрерывное числовое значение.

**Метрика:** одной Accuracy недостаточно, потому что классы распределены неравномерно. Поэтому в работе дополнительно рассматриваются Precision, Recall, F1 и ROC-AUC; ROC-AUC удобна для сравнения способности моделей разделять два класса независимо от конкретного порога.

**Цели работы:**
1. Провести полный EDA датасета Titanic.
2. Обучить и сравнить Logistic Regression и Decision Tree, ориентируясь в том числе на ROC-AUC.
3. Реализовать функцию предсказания для нового пассажира и сохранить обученные модели.


## Теоретическая часть

### EDA (Exploratory Data Analysis)
EDA — это систематическое изучение набора данных до построения моделей. На этом этапе проверяются размерность, типы признаков, пропуски, выбросы и распределения. Также анализируются связи между признаками и целевой переменной. Визуализации помогают заметить закономерности, которые трудно увидеть только по таблицам. Результаты EDA используются для выбора способов очистки и подготовки данных.

### Пропуски (NaN)
Пропуски могут появляться из-за отсутствия информации, ошибок ввода или особенностей сбора данных. Простое удаление строк подходит не всегда, поскольку можно потерять значительную часть наблюдений. Для числовых признаков часто используют медиану, а для категориальных — наиболее частое значение. В Titanic `Age` содержит пропуски, поэтому медиана по группам `Pclass + Sex` позволяет учитывать структуру данных. `Cabin` имеет слишком много пропусков, поэтому в рамках задания этот столбец удаляется.

### Кодирование категорий
Label Encoding заменяет категории числами, например `female → 1`, `male → 0`. Такой подход удобен для бинарного признака `Sex`, если порядок кодирования явно контролируется. One-Hot Encoding создаёт отдельный бинарный столбец для каждой категории и не задаёт искусственного порядка. Поэтому для `Embarked` используется one-hot-представление.

### Масштабирование
StandardScaler приводит признаки примерно к нулевому среднему и единичному стандартному отклонению. MinMaxScaler переводит значения в заданный диапазон, обычно `[0, 1]`. Масштабирование особенно важно для моделей, чувствительных к масштабу признаков, например Logistic Regression. Для Decision Tree масштабирование не требуется, потому что дерево использует пороги по отдельным признакам.

### Feature Engineering
Feature Engineering — создание новых признаков на основе исходных данных. Хороший новый признак может сделать закономерность более явной для модели. В этой работе создаются `Family_Size`, `Is_Alone` и `Age_Group`. Эти признаки позволяют представить размер семьи и возраст пассажира в более удобном для анализа виде.

### Метрики классификации
Accuracy показывает долю правильных предсказаний:

$$Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$$

Precision показывает, какая доля предсказанных положительных классов действительно положительна:

$$Precision = \frac{TP}{TP + FP}$$

Recall показывает, какая доля реальных положительных объектов была найдена:

$$Recall = \frac{TP}{TP + FN}$$

F1 объединяет Precision и Recall:

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

ROC-AUC оценивает качество ранжирования положительных и отрицательных объектов по вероятностям модели. Значение около 0.5 соответствует случайному разделению, а значение ближе к 1 означает лучшее разделение классов.


| Этап | Основной вопрос |
|---|---|
| EDA | Что находится в данных и какие закономерности видны? |
| Предобработка | Как корректно обработать пропуски и категории? |
| Feature Engineering | Какие дополнительные признаки могут помочь? |
| Обучение | Как ведут себя две разные модели? |
| Оценка | Насколько хорошо модели предсказывают выживание? |
| Интерпретация | Какие признаки связаны с результатом? |


## Описание данных

**Источник:** Kaggle — Titanic: Machine Learning from Disaster.

Оригинальный набор содержит `train.csv` на 891 пассажира с известной целевой переменной `Survived` и `test.csv` на 418 пассажиров без целевой переменной.

В работе используются следующие исходные признаки: `PassengerId`, `Pclass`, `Name`, `Sex`, `Age`, `SibSp`, `Parch`, `Ticket`, `Fare`, `Cabin`, `Embarked`. Целевая переменная — `Survived`.

После подготовки для моделей используются: `Pclass`, `Sex_Encoded`, `Age`, `SibSp`, `Parch`, `Fare`, `Family_Size`, `Is_Alone`, `Embarked_Q`, `Embarked_S`.


In [ ]:
# Импортируем основные библиотеки для анализа и моделирования.
from pathlib import Path
import time
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import joblib

# Определяем относительные пути внутри папки module-1-titanic.
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
EXAMPLES_DIR = Path("examples")

DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)
EXAMPLES_DIR.mkdir(exist_ok=True)

# Если CSV ещё не скопированы в репозиторий, пытаемся загрузить официальный по структуре Kaggle источник.
TRAIN_URL = "https://raw.githubusercontent.com/TeamLab/machine_learning_from_scratch_with_python/master/code/ch12/titanic/train.csv"
TEST_URL = "https://raw.githubusercontent.com/TeamLab/machine_learning_from_scratch_with_python/master/code/ch12/titanic/test.csv"

if not (DATA_DIR / "train.csv").exists():
    pd.read_csv(TRAIN_URL).to_csv(DATA_DIR / "train.csv", index=False)
if not (DATA_DIR / "test.csv").exists():
    pd.read_csv(TEST_URL).to_csv(DATA_DIR / "test.csv", index=False)

# Загружаем исходные таблицы.
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

# Проверяем размеры, как требует задание.
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


In [ ]:
# Выводим первые строки для первичного знакомства с данными.
display(train_df.head())

# Проверяем типы столбцов и количество непустых значений.
train_df.info()

# Получаем описательную статистику числовых признаков.
display(train_df.describe().T)


## Подготовка данных и EDA

На этапе EDA сначала проверяются пропуски, распределения и статистические характеристики. Затем строятся обязательные визуализации: распределение целевой переменной, возраст, boxplot, корреляции и показатели выживаемости по ключевым категориям.

Стратегии обработки пропусков:
- `Age` — медиана внутри групп `Pclass + Sex`, потому что возраст связан с социальным классом и полом.
- `Embarked` — наиболее частое значение, поскольку это категориальный признак с небольшим числом пропусков.
- `Cabin` — удаляется, так как в исходном Titanic он содержит очень много пропусков и в рамках задания прямо предусмотрено его удаление.


In [ ]:
# Рассчитываем количество и процент пропусков по каждому столбцу.
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)

missing_table = pd.DataFrame({
    "Missing": missing,
    "Percent": missing_pct
}).sort_values("Missing", ascending=False)

# Показываем таблицу пропусков.
display(missing_table)

# Считаем базовые статистики распределений.
print("Age skew:", train_df["Age"].skew())
print("Age kurtosis:", train_df["Age"].kurtosis())
print("Sex counts:")
display(train_df["Sex"].value_counts())
print("Pclass counts:")
display(train_df["Pclass"].value_counts())


In [ ]:
# Создаём фигуру с шестью графиками EDA.
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Распределение целевой переменной.
sns.countplot(data=train_df, x="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Распределение Survived")
axes[0, 0].set_xlabel("Survived")
axes[0, 0].set_ylabel("Количество")

# 2. Распределение возраста по классам выживания.
sns.histplot(data=train_df, x="Age", hue="Survived", kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Распределение Age по Survived")
axes[0, 1].set_xlabel("Возраст")
axes[0, 1].set_ylabel("Количество")

# 3. Возраст по классу и выживанию.
sns.boxplot(data=train_df, x="Pclass", y="Age", hue="Survived", ax=axes[0, 2])
axes[0, 2].set_title("Age по Pclass и Survived")
axes[0, 2].set_xlabel("Класс")
axes[0, 2].set_ylabel("Возраст")

# 4. Корреляционная матрица числовых признаков.
corr = train_df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="Blues", ax=axes[1, 0])
axes[1, 0].set_title("Корреляция числовых признаков")

# 5. Выживаемость по полу.
survival_sex = train_df.groupby("Sex")["Survived"].mean().reset_index()
sns.barplot(data=survival_sex, x="Sex", y="Survived", ax=axes[1, 1])
axes[1, 1].set_title("Доля выживших по полу")
axes[1, 1].set_xlabel("Пол")
axes[1, 1].set_ylabel("Доля выживших")

# 6. Выживаемость по классу.
survival_class = train_df.groupby("Pclass")["Survived"].mean().reset_index()
sns.barplot(data=survival_class, x="Pclass", y="Survived", ax=axes[1, 2])
axes[1, 2].set_title("Доля выживших по классу")
axes[1, 2].set_xlabel("Класс")
axes[1, 2].set_ylabel("Доля выживших")

# Сохраняем общий рисунок для папки examples.
plt.tight_layout()
plt.savefig(EXAMPLES_DIR / "eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()


### Наблюдения по EDA

1. В целевой переменной присутствуют два класса, причём невыживших больше, чем выживших.
2. Возраст имеет пропуски, поэтому перед обучением его необходимо заполнить.
3. Между полом и выживаемостью наблюдается заметная разница.
4. Класс билета также связан с долей выживших: группы пассажиров разных классов отличаются по результату.
5. `Cabin` содержит много пропусков, поэтому для базовой модели его целесообразно исключить.
6. `Family_Size` и `Is_Alone` позволяют дополнительно учитывать семейный контекст пассажира.


In [ ]:
# Копируем исходные данные, чтобы сохранить оригинальные таблицы без изменений.
train_clean = train_df.copy()
test_clean = test_df.copy()

# Заполняем Age медианой внутри групп Pclass + Sex.
for frame in [train_clean, test_clean]:
    frame["Age"] = frame.groupby(["Pclass", "Sex"])["Age"].transform(
        lambda s: s.fillna(s.median())
    )
    frame["Age"] = frame["Age"].fillna(train_clean["Age"].median())

# Заполняем Embarked модой и удаляем Cabin согласно заданию.
train_clean["Embarked"] = train_clean["Embarked"].fillna(train_clean["Embarked"].mode()[0])
test_clean["Embarked"] = test_clean["Embarked"].fillna(train_clean["Embarked"].mode()[0])
train_clean = train_clean.drop(columns=["Cabin"])
test_clean = test_clean.drop(columns=["Cabin"])

# Создаём семейные признаки.
for frame in [train_clean, test_clean]:
    frame["Family_Size"] = frame["SibSp"] + frame["Parch"] + 1
    frame["Is_Alone"] = (frame["Family_Size"] == 1).astype(int)
    frame["Age_Group"] = pd.cut(
        frame["Age"],
        bins=[0, 12, 18, 35, 60, np.inf],
        labels=["Child", "Teen", "Adult", "Middle_Age", "Senior"]
    )

# Кодируем Sex через LabelEncoder, как требует задание.
le_sex = LabelEncoder()
le_sex.fit(["female", "male"])
train_clean["Sex_Encoded"] = le_sex.transform(train_clean["Sex"])
test_clean["Sex_Encoded"] = le_sex.transform(test_clean["Sex"])

# Кодируем Embarked через one-hot и фиксируем одинаковые столбцы для train/test.
train_clean = pd.get_dummies(train_clean, columns=["Embarked"], prefix="Embarked")
test_clean = pd.get_dummies(test_clean, columns=["Embarked"], prefix="Embarked")

for col in ["Embarked_C", "Embarked_Q", "Embarked_S"]:
    if col not in train_clean:
        train_clean[col] = 0
    if col not in test_clean:
        test_clean[col] = 0

# Приводим one-hot признаки к целочисленному типу.
for col in ["Embarked_C", "Embarked_Q", "Embarked_S"]:
    train_clean[col] = train_clean[col].astype(int)
    test_clean[col] = test_clean[col].astype(int)

print("Пропуски после обработки:")
display(train_clean.isnull().sum().sort_values(ascending=False).head(10))


## Построение и обучение моделей

Для сравнения используются две обязательные модели:

- **Logistic Regression** — простая интерпретируемая линейная модель. Для неё применяется `StandardScaler`.
- **Decision Tree** — нелинейная модель, которая не требует масштабирования и позволяет анализировать важность признаков.

Для честного сравнения данные делятся на train/test с `stratify=y`, чтобы доля классов была сохранена. `random_state=42` обеспечивает воспроизводимость.


In [ ]:
# Определяем признаки в том же порядке, который будет сохранён для инференса.
feature_cols = [
    "Pclass", "Sex_Encoded", "Age", "SibSp", "Parch",
    "Fare", "Family_Size", "Is_Alone", "Embarked_Q", "Embarked_S"
]

# Формируем матрицу признаков и целевую переменную.
X = train_clean[feature_cols].copy()
y = train_clean["Survived"].copy()

# Делим данные на обучающую и тестовую выборки со стратификацией.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Масштабируем признаки только для Logistic Regression.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Обучаем Logistic Regression и измеряем время.
lr_model = LogisticRegression(penalty="l2", C=1.0, max_iter=1000, random_state=42)
start = time.time()
lr_model.fit(X_train_scaled, y_train)
lr_time = time.time() - start

# Обучаем Decision Tree без масштабирования и измеряем время.
dt_model = DecisionTreeClassifier(
    max_depth=5, min_samples_split=5, min_samples_leaf=2, random_state=42
)
start = time.time()
dt_model.fit(X_train, y_train)
dt_time = time.time() - start

print(f"Время обучения Logistic Regression: {lr_time:.4f} сек.")
print(f"Время обучения Decision Tree: {dt_time:.4f} сек.")


## Оценка качества

Для обеих моделей рассчитываются Accuracy, Precision, Recall, F1 и ROC-AUC. Также строятся матрицы ошибок и общая ROC-кривая.

ROC-AUC рассчитывается по вероятностям положительного класса, а остальные основные метрики — по бинарным предсказаниям.


In [ ]:
# Получаем классы и вероятности для Logistic Regression.
y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Получаем классы и вероятности для Decision Tree.
y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

# Собираем метрики в одну таблицу.
metrics_table = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_dt)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt)
    ],
    "F1": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_dt)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_lr),
        roc_auc_score(y_test, y_proba_dt)
    ],
    "Training_Time_sec": [lr_time, dt_time]
})

# Округляем значения для удобного отображения.
display(metrics_table.round(4))

# Выводим подробные classification reports.
print("Logistic Regression:")
print(classification_report(y_test, y_pred_lr))
print("Decision Tree:")
print(classification_report(y_test, y_pred_dt))


In [ ]:
# Создаём две матрицы ошибок рядом.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Матрица ошибок Logistic Regression.
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(
    cm_lr, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Не выжил", "Выжил"],
    yticklabels=["Не выжил", "Выжил"],
    ax=axes[0]
)
axes[0].set_title("Logistic Regression")
axes[0].set_xlabel("Предсказано")
axes[0].set_ylabel("Истинно")

# Матрица ошибок Decision Tree.
cm_dt = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(
    cm_dt, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Не выжил", "Выжил"],
    yticklabels=["Не выжил", "Выжил"],
    ax=axes[1]
)
axes[1].set_title("Decision Tree")
axes[1].set_xlabel("Предсказано")
axes[1].set_ylabel("Истинно")

# Сохраняем рисунок для репозитория.
plt.tight_layout()
plt.savefig(EXAMPLES_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Строим ROC-кривые двух моделей на одном графике.
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_proba_dt)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_lr, tpr_lr,
    label=f"Logistic Regression (AUC={roc_auc_score(y_test, y_proba_lr):.3f})",
    lw=2
)
plt.plot(
    fpr_dt, tpr_dt,
    label=f"Decision Tree (AUC={roc_auc_score(y_test, y_proba_dt):.3f})",
    lw=2
)
plt.plot([0, 1], [0, 1], "k--", label="Случайный классификатор")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривые")
plt.legend()
plt.grid(alpha=0.3)

# Сохраняем график ROC-кривых.
plt.savefig(EXAMPLES_DIR / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()


### Анализ качества

После запуска ячейки с метриками значения автоматически появятся в таблице. Сравнивать модели следует сразу по нескольким показателям, а не по одной Accuracy.

Особое внимание уделяется ROC-AUC и F1: ROC-AUC характеризует качество разделения классов по вероятностям, а F1 учитывает одновременно Precision и Recall. Если дерево показывает заметно более высокое качество на обучении, но слабее обобщает данные, это может быть признаком переобучения; ограничение `max_depth=5` уменьшает этот риск.


## Интерпретация результатов

Коэффициенты Logistic Regression показывают направление влияния стандартизированных признаков на логарифм шансов положительного класса. Положительный коэффициент связан с увеличением шансов класса `Survived=1`, отрицательный — с уменьшением при прочих равных условиях.

Feature Importance дерева показывает, насколько часто и насколько эффективно признаки используются для разделения объектов в построенной структуре дерева. Эти показатели не следует трактовать как причинно-следственные связи.


In [ ]:
# Создаём таблицу коэффициентов Logistic Regression.
coefficients = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lr_model.coef_[0]
}).sort_values("Coefficient", ascending=False)

# Показываем коэффициенты в порядке от положительных к отрицательным.
display(coefficients.round(4))

plt.figure(figsize=(10, 6))
plt.barh(coefficients["Feature"], coefficients["Coefficient"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Коэффициент")
plt.title("Коэффициенты Logistic Regression")
plt.tight_layout()
plt.show()

# Создаём таблицу важности признаков Decision Tree.
importances = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": dt_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importances.round(4))

plt.figure(figsize=(10, 6))
sns.barplot(data=importances, x="Importance", y="Feature")
plt.title("Feature Importance: Decision Tree")
plt.tight_layout()
plt.show()


### Наблюдения и прикладные выводы

- Пол, класс билета и стоимость билета обычно оказываются среди наиболее информативных признаков в Titanic-моделях.
- Более высокий класс билета связан с другой социальной и экономической группой пассажиров, поэтому `Pclass` может хорошо разделять наблюдения.
- Семейные признаки добавляют информацию о том, путешествовал ли пассажир один и какого размера была его семья.
- Коэффициенты и важности признаков описывают работу конкретной модели на этих данных; они не доказывают причинность.


## Функция предсказания

Функция ниже повторяет те же шаги Feature Engineering и кодирования, которые использовались при обучении. Это важно: новый объект должен иметь тот же набор и порядок признаков, что и обучающая выборка.

Функция возвращает класс (`0` или `1`) и вероятность выживания.


In [ ]:
# Определяем функцию подготовки и предсказания для одного нового пассажира.
def predict_passenger(passenger_data, model, scaler=None, feature_cols=None, use_scaling=True):
    data = passenger_data.copy()

    # Создаём семейные признаки тем же способом, что и при обучении.
    data["Family_Size"] = data["SibSp"] + data["Parch"] + 1
    data["Is_Alone"] = int(data["Family_Size"] == 1)

    # Кодируем пол и порт отправления так же, как в обучающем наборе.
    data["Sex_Encoded"] = 1 if data["Sex"] == "female" else 0
    data["Embarked_Q"] = int(data["Embarked"] == "Q")
    data["Embarked_S"] = int(data["Embarked"] == "S")

    # Собираем признаки строго в заданном порядке.
    X_new = pd.DataFrame([{
        "Pclass": data["Pclass"],
        "Sex_Encoded": data["Sex_Encoded"],
        "Age": data["Age"],
        "SibSp": data["SibSp"],
        "Parch": data["Parch"],
        "Fare": data["Fare"],
        "Family_Size": data["Family_Size"],
        "Is_Alone": data["Is_Alone"],
        "Embarked_Q": data["Embarked_Q"],
        "Embarked_S": data["Embarked_S"]
    }])[feature_cols]

    # Масштабируем только для Logistic Regression.
    if use_scaling:
        X_new = scaler.transform(X_new)

    # Получаем класс и вероятность положительного класса.
    prediction = int(model.predict(X_new)[0])
    probability = float(model.predict_proba(X_new)[0, 1])

    return prediction, probability


In [ ]:
# Создаём пять пассажиров с разными профилями, как требует задание.
examples = [
    {"Pclass": 1, "Sex": "female", "Age": 25, "SibSp": 1, "Parch": 0, "Fare": 100, "Embarked": "C"},
    {"Pclass": 3, "Sex": "male", "Age": 30, "SibSp": 0, "Parch": 0, "Fare": 7.25, "Embarked": "S"},
    {"Pclass": 2, "Sex": "female", "Age": 5, "SibSp": 3, "Parch": 1, "Fare": 30, "Embarked": "S"},
    {"Pclass": 1, "Sex": "male", "Age": 60, "SibSp": 0, "Parch": 0, "Fare": 200, "Embarked": "C"},
    {"Pclass": 3, "Sex": "female", "Age": 22, "SibSp": 0, "Parch": 0, "Fare": 8, "Embarked": "Q"}
]

# Для каждого пассажира сравниваем две модели.
prediction_rows = []
for i, passenger in enumerate(examples, start=1):
    pred_lr, prob_lr = predict_passenger(
        passenger, lr_model, scaler, feature_cols, use_scaling=True
    )
    pred_dt, prob_dt = predict_passenger(
        passenger, dt_model, None, feature_cols, use_scaling=False
    )

    prediction_rows.append({
        "Example": i,
        "LR prediction": pred_lr,
        "LR probability": round(prob_lr, 4),
        "DT prediction": pred_dt,
        "DT probability": round(prob_dt, 4),
        "Models agree": pred_lr == pred_dt
    })

# Показываем результаты пяти примеров.
display(pd.DataFrame(prediction_rows))


## Сохранение модели

Сохраняются обе модели, `StandardScaler`, `LabelEncoder`, список признаков и рассчитанные метрики. Это позволяет воспроизводимо использовать модель после завершения ноутбука.


In [ ]:
# Сохраняем обученные модели и трансформеры через joblib.
joblib.dump(lr_model, MODELS_DIR / "lr_model.pkl")
joblib.dump(dt_model, MODELS_DIR / "dt_model.pkl")
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(le_sex, MODELS_DIR / "le_sex.pkl")

# Сохраняем порядок признаков в JSON.
with open(MODELS_DIR / "feature_cols.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

# Формируем словарь метрик для двух моделей.
metrics = {}
for _, row in metrics_table.iterrows():
    metrics[row["Model"]] = {
        "accuracy": float(row["Accuracy"]),
        "precision": float(row["Precision"]),
        "recall": float(row["Recall"]),
        "f1": float(row["F1"]),
        "roc_auc": float(row["ROC-AUC"]),
        "training_time_sec": float(row["Training_Time_sec"])
    }

# Сохраняем метрики.
with open(MODELS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Модели и метаданные сохранены в папку models/.")


## Загрузка модели из репозитория

После создания GitHub-репозитория значение `BASE_URL` можно заменить на raw-ссылку вида:

`https://raw.githubusercontent.com/USERNAME/ml-course-dremin/main/module-1-titanic`

Следующая ячейка загружает модель без переобучения. До публикации репозитория она будет работать только после того, как файлы появятся по указанному URL.


In [ ]:
# После публикации GitHub-репозитория замените USERNAME на свой логин.
import requests
from io import BytesIO

BASE_URL = "https://raw.githubusercontent.com/USERNAME/ml-course-dremin/main/module-1-titanic"

# Загружаем Logistic Regression из GitHub.
model_url = f"{BASE_URL}/models/lr_model.pkl"
response = requests.get(model_url, timeout=30)
response.raise_for_status()
lr_model_loaded = joblib.load(BytesIO(response.content))

# Загружаем scaler и список признаков.
scaler_url = f"{BASE_URL}/models/scaler.pkl"
scaler_loaded = joblib.load(BytesIO(requests.get(scaler_url, timeout=30).content))

features_url = f"{BASE_URL}/models/feature_cols.json"
feature_cols_loaded = requests.get(features_url, timeout=30).json()

print("Модель загружена из GitHub.")
print("Количество признаков:", len(feature_cols_loaded))


In [ ]:
# Проверяем загруженную модель на трёх пассажирах.
for passenger in examples[:3]:
    original = predict_passenger(
        passenger, lr_model, scaler, feature_cols, use_scaling=True
    )
    loaded = predict_passenger(
        passenger, lr_model_loaded, scaler_loaded, feature_cols_loaded, use_scaling=True
    )
    print("Совпадает:", original == loaded, "|", original, "|", loaded)


## Выводы

### Что получилось
- Проведён EDA исходного Titanic.
- Обработаны пропуски `Age` и `Embarked`, удалён `Cabin`.
- Созданы `Family_Size`, `Is_Alone` и `Age_Group`.
- Обучены Logistic Regression и Decision Tree.
- Рассчитаны Accuracy, Precision, Recall, F1 и ROC-AUC.
- Реализована функция `predict_passenger()` и пять демонстрационных примеров.
- Модели и вспомогательные объекты сохраняются через `joblib`.

### Трудности
- В `Age` есть пропущенные значения.
- `Cabin` содержит большое количество пропусков.
- Целевая переменная распределена неравномерно.
- Decision Tree может переобучаться, поэтому глубина дерева ограничена.

### Возможные улучшения
1. Извлечь `Title` из `Name`.
2. Добавить взаимодействия признаков, например `Pclass × Sex`.
3. Применить методы балансировки классов, например SMOTE.
4. Сравнить с Random Forest и Gradient Boosting.
5. Подобрать гиперпараметры через `GridSearchCV`.
6. Исследовать преобразование `Fare`, например `np.log1p(Fare)`.


## Источники

1. Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
2. McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
3. Kaggle. *Titanic — Machine Learning from Disaster*.
4. Scikit-learn Documentation. https://scikit-learn.org/stable/
5. Pandas Documentation. https://pandas.pydata.org/docs/
6. Matplotlib Documentation. https://matplotlib.org/stable/
7. Seaborn Documentation. https://seaborn.pydata.org/


## Приложение

Ниже приведён список зависимостей проекта. Полный рабочий код находится в ячейках этого ноутбука в порядке выполнения.

```text
# module-1-titanic/requirements.txt
numpy==1.24.0
pandas==2.0.0
matplotlib==3.7.0
seaborn==0.12.0
scikit-learn==1.3.0
joblib==1.3.0
requests==2.31.0
```

Все основные этапы разделены на отдельные ячейки, а комментарии в коде поясняют назначение выполняемых операций.
